# SNLI-VE VLM Baselines — Zero-shot, Linear Probe, MLP Probe, Fine-tuned CLIP
**Project**: Quantum Kernels on CLIP/BERT Embeddings — ICLR Extension

**Purpose:** surround the quantum kernel result with the VLM baselines a reviewer
will demand. Four baselines, weakest to strongest:

1. **CLIP zero-shot** — no training, cosine similarity to class text templates (the floor)
2. **CLIP linear probe** — single linear layer on frozen joint embeddings
3. **MLP probe** — 2-layer neural net on frozen joint embeddings
4. **Fine-tuned CLIP** — unfreeze the vision encoder, train end-to-end (the ceiling)

**The framing for the paper:**
Quantum and classical kernel SVMs both operate on FROZEN embeddings. The probes
(linear, MLP) are the standard frozen-representation baselines and sit in the same
tier — these are the fair comparison. Zero-shot is the lower reference, fine-tuned
CLIP is the upper reference (expected to win, since it adapts the encoder itself).

**Inputs:** `snlive_embeddings.npz`, `snlive_raw.npz` (from Week 2 SNLI-VE)
**Needs:** the original images + texts for zero-shot and fine-tuning — re-loads dataset
**Runtime:** ~5 min (probes) + ~40 min (fine-tuning, GPU strongly recommended)

In [ ]:
!pip install datasets transformers torch Pillow scikit-learn numpy --quiet
import numpy as np, time, os, pickle, shutil, json
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import CLIPModel, CLIPProcessor
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
if DEVICE == 'cpu':
    print("WARNING: fine-tuning CLIP on CPU is very slow. Switch to a GPU runtime:")
    print("  Runtime > Change runtime type > T4 GPU")

---
## Step 0: Storage + helpers

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)
TAG = 'snlive'
print(f"Save dir: {SAVE_DIR}")

def _persist_and_download(path):
    if os.path.abspath(SAVE_DIR) != os.getcwd():
        shutil.copy(path, os.path.join(SAVE_DIR, os.path.basename(path)))
        print(f"  saved -> {SAVE_DIR}/{os.path.basename(path)}")
    try:
        from google.colab import files
        files.download(path)
        print(f"  download -> {os.path.basename(path)}")
    except Exception as e:
        print(f"  (download skipped: {type(e).__name__} — saved above)")

def checkpoint(filename, save_fn): save_fn(filename); _persist_and_download(filename)
def save_figure(fig, name):
    for ext in ('png','pdf'):
        p = f"{name}.{ext}"
        fig.savefig(p, bbox_inches='tight', dpi=150); _persist_and_download(p)
def _find(fname):
    dp = os.path.join(SAVE_DIR, fname)
    return dp if os.path.exists(dp) else fname
print("Helpers ready.")

---
## Step 1: Load frozen embeddings (for probes) + raw 1024-dim (for splitting img/text)

The joint embedding is [image_512 ; text_512]. We can split it back to access
the image half for the image-only zero-shot baseline.

In [ ]:
# Final 8-dim PCA embeddings used by the kernels (for fair-tier probes we use the
# FULL 1024-dim joint embedding, not the 8-dim — probes can handle full dimensionality)
raw = np.load(_find('snlive_raw.npz'))
train_emb_full = raw['train_emb']   # (N, 1024) joint
test_emb_full  = raw['test_emb']
train_labels   = raw['train_labels']
test_labels    = raw['test_labels']

# subsample to the same 1000/200 the kernels used (same seed)
idx_tr = np.random.RandomState(42).choice(len(train_emb_full), 1000, replace=False)
idx_te = np.random.RandomState(42).choice(len(test_emb_full),  200,  replace=False)
Xtr_full, ytr = train_emb_full[idx_tr], train_labels[idx_tr]
Xte_full, yte = test_emb_full[idx_te],  test_labels[idx_te]

# split joint -> image half (first 512) and text half (last 512)
Xtr_img, Xtr_txt = Xtr_full[:, :512], Xtr_full[:, 512:]
Xte_img, Xte_txt = Xte_full[:, :512], Xte_full[:, 512:]

LABELS = {0:'entailment', 1:'neutral', 2:'contradiction'}
def mf1(yt, yp): return float(f1_score(yt, yp, average='macro'))

print(f"Full joint embeddings: train {Xtr_full.shape}, test {Xte_full.shape}")
print(f"Image half: {Xtr_img.shape}  Text half: {Xtr_txt.shape}")
print(f"Class balance (train): {dict(zip(*np.unique(ytr, return_counts=True)))}")

baseline_results = {}

---
## Baseline 1: CLIP Zero-Shot (no training — the floor)

For visual entailment, zero-shot is approximate: we compare the image embedding
to text-template embeddings for each class. This is a rough reference, since true
visual entailment depends on the hypothesis, not just the image. We report it as a
lower bound on what CLIP "knows" without any task supervision.

In [ ]:
# Use the stored image-half embeddings (already L2-normalised from Week 2)
clip_model_zs = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
clip_proc_zs  = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model_zs.eval()

templates = [
    "the image clearly shows this is true",       # entailment
    "the image does not show whether this is true",# neutral
    "the image shows this is false",               # contradiction
]
with torch.no_grad():
    tok = clip_proc_zs.tokenizer(templates, return_tensors='pt', padding=True).to(DEVICE)
    tmpl_emb = clip_model_zs.get_text_features(**tok)
    tmpl_emb = (tmpl_emb / torch.linalg.norm(tmpl_emb, dim=-1, keepdim=True)).cpu().numpy()

# cosine similarity between each test image embedding and the 3 templates
sims = Xte_img @ tmpl_emb.T          # (200, 3)
zs_pred = np.argmax(sims, axis=1)
baseline_results['CLIP Zero-shot'] = {
    'accuracy': float(accuracy_score(yte, zs_pred)),
    'f1': mf1(yte, zs_pred)
}
print(f"CLIP Zero-shot: acc={baseline_results['CLIP Zero-shot']['accuracy']:.4f}  "
      f"macro-F1={baseline_results['CLIP Zero-shot']['f1']:.4f}")
print("(Reference floor — uses image only, no hypothesis interaction)")

---
## Baseline 2: CLIP Linear Probe (frozen embeddings + single linear layer)

The standard representation-learning baseline. Logistic regression on the full
1024-dim joint embedding is exactly a linear probe. This sits in the SAME tier as
your quantum and classical kernel SVMs — the fair comparison.

In [ ]:
lin = LogisticRegression(max_iter=2000, C=1.0, random_state=42)
lin.fit(Xtr_full, ytr)
lp = lin.predict(Xte_full)
baseline_results['Linear Probe'] = {
    'accuracy': float(accuracy_score(yte, lp)),
    'f1': mf1(yte, lp)
}
print(f"Linear Probe (1024-dim joint): acc={baseline_results['Linear Probe']['accuracy']:.4f}  "
      f"macro-F1={baseline_results['Linear Probe']['f1']:.4f}")

---
## Baseline 3: MLP Probe (frozen embeddings + 2-layer neural net)

A stronger frozen-representation baseline. Still operates on the same frozen
1024-dim joint embeddings — same tier as the kernels and linear probe, but with
nonlinear capacity.

In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(256, 64), max_iter=500,
                    early_stopping=True, random_state=42)
mlp.fit(Xtr_full, ytr)
mp = mlp.predict(Xte_full)
baseline_results['MLP Probe'] = {
    'accuracy': float(accuracy_score(yte, mp)),
    'f1': mf1(yte, mp)
}
print(f"MLP Probe (256-64 on 1024-dim): acc={baseline_results['MLP Probe']['accuracy']:.4f}  "
      f"macro-F1={baseline_results['MLP Probe']['f1']:.4f}")

---
## Baseline 4: Fine-tuned CLIP (unfreeze + train end-to-end — the ceiling)

This is the strong upper-reference baseline. We re-load the actual images and
hypotheses, attach a classification head, and fine-tune the CLIP joint encoder
end-to-end. This adapts the encoder itself, so it has far more capacity than any
frozen-embedding method. It is EXPECTED to win — it is the ceiling, not a
competitor to the quantum kernel.

**GPU strongly recommended.** On CPU this will be extremely slow.

In [ ]:
# Re-load the same SNLI-VE samples (images + texts) used in Week 2
from datasets import load_dataset
print("Re-loading SNLI-VE for fine-tuning...")
dataset = load_dataset("HuggingFaceM4/SNLI-VE")
cols = dataset['train'].column_names
img_col = 'image' if 'image' in cols else 'premise'
txt_col = 'hypothesis' if 'hypothesis' in cols else 'caption'

def load_balanced(split, n_per_class):
    imgs, txts, lbls = [], [], []
    counts = {0:0,1:0,2:0}
    for s in dataset[split]:
        l = s['label']
        if l not in [0,1,2] or counts[l] >= n_per_class: continue
        imgs.append(s[img_col]); txts.append(s[txt_col]); lbls.append(l); counts[l]+=1
        if all(v>=n_per_class for v in counts.values()): break
    return imgs, txts, np.array(lbls)

ft_train_imgs, ft_train_txts, ft_train_lbls = load_balanced('train', 350)  # ~1050
ft_test_imgs,  ft_test_txts,  ft_test_lbls  = load_balanced('test', 70)    # ~210
print(f"Fine-tune train: {len(ft_train_imgs)}  test: {len(ft_test_imgs)}")

In [ ]:
# Fine-tuning model: CLIP joint features -> classification head, encoder unfrozen
class CLIPEntailment(nn.Module):
    def __init__(self, clip_model, n_classes=3):
        super().__init__()
        self.clip = clip_model
        self.head = nn.Sequential(
            nn.Linear(1024, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, n_classes)
        )
    def forward(self, pixel_values, input_ids, attention_mask):
        vis = self.clip.vision_model(pixel_values=pixel_values).pooler_output
        img = self.clip.visual_projection(vis)
        img = img / torch.linalg.norm(img, dim=-1, keepdim=True)
        txt = self.clip.get_text_features(input_ids=input_ids, attention_mask=attention_mask)
        txt = txt / torch.linalg.norm(txt, dim=-1, keepdim=True)
        joint = torch.cat([img, txt], dim=-1)
        return self.head(joint)

clip_base = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
proc      = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model_ft  = CLIPEntailment(clip_base).to(DEVICE)
print(f"Fine-tune model ready. Trainable params: {sum(p.numel() for p in model_ft.parameters() if p.requires_grad):,}")

In [ ]:
def make_batches(imgs, txts, lbls, bs=16):
    for i in range(0, len(imgs), bs):
        ib, tb, lb = imgs[i:i+bs], txts[i:i+bs], lbls[i:i+bs]
        img_in = proc(images=ib, return_tensors='pt')
        txt_in = proc.tokenizer(tb, return_tensors='pt', padding=True, truncation=True, max_length=77)
        yield (img_in['pixel_values'].to(DEVICE),
               txt_in['input_ids'].to(DEVICE),
               txt_in['attention_mask'].to(DEVICE),
               torch.tensor(lb, dtype=torch.long).to(DEVICE))

opt = torch.optim.AdamW(model_ft.parameters(), lr=1e-5)
lossf = nn.CrossEntropyLoss()
EPOCHS = 3

print(f"Fine-tuning for {EPOCHS} epochs (~{EPOCHS*10}-{EPOCHS*20} min on GPU)...")
for epoch in range(EPOCHS):
    model_ft.train(); total = 0; t0 = time.time()
    for pv, ii, am, yb in make_batches(ft_train_imgs, ft_train_txts, ft_train_lbls):
        opt.zero_grad()
        out = model_ft(pv, ii, am)
        loss = lossf(out, yb)
        loss.backward(); opt.step()
        total += loss.item()
    print(f"  Epoch {epoch+1}/{EPOCHS}  loss={total:.3f}  ({(time.time()-t0)/60:.1f} min)")

# Evaluate
model_ft.eval(); preds = []
with torch.no_grad():
    for pv, ii, am, yb in make_batches(ft_test_imgs, ft_test_txts, ft_test_lbls):
        preds.extend(model_ft(pv, ii, am).argmax(-1).cpu().numpy())
preds = np.array(preds)
baseline_results['Fine-tuned CLIP'] = {
    'accuracy': float(accuracy_score(ft_test_lbls, preds)),
    'f1': mf1(ft_test_lbls, preds)
}
print(f"\nFine-tuned CLIP: acc={baseline_results['Fine-tuned CLIP']['accuracy']:.4f}  "
      f"macro-F1={baseline_results['Fine-tuned CLIP']['f1']:.4f}")

---
## Step: Load quantum + classical kernel results to complete the picture

In [ ]:
# pull in the kernel results from Week 3 SNLI-VE for the full comparison table
kernel_results = {}
try:
    with open(_find('snlive_results_week3.json')) as f:
        wk3 = json.load(f)
    kernel_results = wk3.get('all_results', {})
    print("Loaded Week 3 kernel results (quantum + classical SVMs).")
except:
    print("snlive_results_week3.json not found — table will show baselines only.")
    print("Run Week 3 SNLI-VE first to include quantum/classical kernels.")

---
## Full results table + figure

In [ ]:
# Combine everything into one ranked table
combined = dict(baseline_results)
for k, v in kernel_results.items():
    combined[k] = v   # Linear/RBF/Polynomial SVM + Quantum (IQP)

print("="*60)
print("SNLI-VE — FULL BASELINE COMPARISON (macro F1)")
print("="*60)
print(f"{'Method':<22} {'Accuracy':>10} {'Macro-F1':>10}  {'Tier'}")
print("-"*60)
tiers = {
    'CLIP Zero-shot': 'floor',
    'Linear Probe': 'frozen', 'MLP Probe': 'frozen',
    'Linear': 'frozen-kernel', 'RBF': 'frozen-kernel', 'Polynomial': 'frozen-kernel',
    'Quantum (IQP)': 'frozen-kernel',
    'Fine-tuned CLIP': 'ceiling',
}
for name in sorted(combined, key=lambda k: combined[k]['f1']):
    v = combined[name]
    print(f"{name:<22} {v['accuracy']:>10.4f} {v['f1']:>10.4f}  {tiers.get(name,'')}")
print("="*60)

# Figure — colour by tier
fig, ax = plt.subplots(figsize=(12, 6))
order = sorted(combined, key=lambda k: combined[k]['f1'])
f1s   = [combined[n]['f1'] for n in order]
tier_color = {'floor':'#BBBBBB','frozen':'#5B8DB8','frozen-kernel':'#2D9C4A',
              'ceiling':'#9B59B6'}
colors = [('#D95F02' if n=='Quantum (IQP)' else tier_color.get(tiers.get(n,''), '#5B8DB8')) for n in order]
bars = ax.barh(order, f1s, color=colors, edgecolor='white')
ax.set_xlabel('Macro F1'); ax.set_xlim(0, 1.0)
ax.set_title('SNLI-VE Visual Entailment — full baseline ladder\n(Quantum IQP in orange; fine-tuned CLIP is the ceiling)')
for b, v in zip(bars, f1s):
    ax.text(v+0.008, b.get_y()+b.get_height()/2, f'{v:.3f}', va='center', fontsize=10)
plt.tight_layout(); plt.show()
save_figure(fig, 'snlive_full_baseline_ladder')

---
## Save

In [ ]:
out = {'dataset':'snlive', 'baselines': baseline_results,
       'kernel_results': kernel_results, 'combined': combined,
       'tiers': tiers}
checkpoint('snlive_all_baselines.json', lambda p: json.dump(out, open(p,'w'), indent=2))
print("\n=== SNLI-VE BASELINES DONE ===")
print("Tiers: zero-shot (floor) < frozen probes/kernels < fine-tuned CLIP (ceiling)")
print("The quantum-vs-classical comparison lives in the frozen-kernel tier.")

---
## Done — the complete baseline ladder

### What you now have for SNLI-VE:
| Tier | Method | What it shows |
|------|--------|---------------|
| Floor | CLIP zero-shot | What CLIP knows with no training |
| Frozen | Linear probe | Standard frozen-rep baseline |
| Frozen | MLP probe | Nonlinear frozen-rep baseline |
| Frozen-kernel | Linear/RBF/Poly SVM | Your classical kernels |
| Frozen-kernel | Quantum IQP | Your quantum kernel |
| Ceiling | Fine-tuned CLIP | Full end-to-end VLM |

### The honest framing for the paper:
"Quantum and classical kernels operate on frozen CLIP embeddings. We benchmark
against standard frozen-representation probes (linear, MLP) in the same tier, and
report CLIP zero-shot and fully fine-tuned CLIP as lower and upper references. The
quantum-vs-classical comparison is conducted within the frozen-representation
regime, which reflects the practical setting where the encoder is fixed."

### This preempts the reviewer question:
"Why not fine-tune?" — you did, and you report it. It wins, as expected, because
it adapts the encoder. Your contribution is about the frozen-kernel tier, clearly
delineated.